<!-- NB_DOC_INTRO_V1 -->
# FastAPI — API moderne async

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**FastAPI** est un framework Python moderne pour APIs HTTP basee sur Starlette (ASGI async) + Pydantic v2 (validation des types). C'est l'**alternative recommandee a Flask en 2025** pour tout nouveau projet.

Avantages cles :
- **Async natif** (`async def`) → bien meilleur sur I/O bound (DB, HTTP outbound)
- **Validation type-hint** automatique via Pydantic v2 → 0 boilerplate
- **OpenAPI 3 auto-genere** → Swagger UI + ReDoc gratuits
- **Performance ~ 3-5x Flask** (TechEmpower benchmarks)
- **Type hints = source de verite** (doc + validation + IDE help)

Ce notebook execute **tous les endpoints** in-process via `TestClient` (pas besoin de serveur reel). Couvre : modeles Pydantic, validation, async, Depends (injection de deps), auth OAuth2/JWT, tests automatises, serving ML, deploiement Docker/Nginx.

Versions visees : `fastapi >= 0.110`, `pydantic >= 2.6`, `uvicorn >= 0.27`.

## Plan

1. Installation et hello world
2. Modeles Pydantic v2 (validation forte)
3. API CRUD complete (livres) + test
4. Async vs sync (quand utiliser quoi)
5. Depends — injection de dependances
6. OAuth2 + JWT (auth pattern recommande)
7. Tests automatises (pytest + TestClient)
8. Serving ML — pattern lifespan + state
9. Deploiement (Gunicorn, Docker, Nginx)
10. Comparatif Flask vs FastAPI
11. Pieges et anti-patterns
12. References


## 1. Installation et hello world

FastAPI fonctionne immediatement en CLI via `uvicorn`, mais on va l'utiliser ici via `TestClient` (sans serveur).


In [ ]:
# pip install -q fastapi 'uvicorn[standard]' pydantic httpx
import fastapi
import pydantic
from fastapi import FastAPI
from fastapi.testclient import TestClient

print(f"FastAPI  : {fastapi.__version__}")
print(f"Pydantic : {pydantic.VERSION}")

# Hello world
app = FastAPI(title="Demo", version="1.0.0")

@app.get("/")
def root():
    return {"message": "Hello FastAPI"}

@app.get("/items/{item_id}")
def get_item(item_id: int):
    # item_id est cast en int automatiquement par FastAPI
    # Si l'utilisateur passe ?item_id=abc → 422 Unprocessable Entity automatique
    return {"item_id": item_id, "name": f"Item {item_id}"}

client = TestClient(app)
print(f"\nGET /          : {client.get('/').json()}")
print(f"GET /items/42  : {client.get('/items/42').json()}")
print(f"GET /items/abc : status {client.get('/items/abc').status_code} (422 attendu)")

## 2. Modeles Pydantic v2 — validation forte

Source unique de verite pour : **validation**, **serialisation**, **doc OpenAPI**.

Pydantic v2 (rewrite Rust) est **5-50x plus rapide** que v1.


In [ ]:
from pydantic import BaseModel, Field, EmailStr, ConfigDict, field_validator
from datetime import datetime
from typing import Optional

class Book(BaseModel):
    """Modele d'entree d'un livre."""
    model_config = ConfigDict(str_strip_whitespace=True)   # strip whitespace auto

    title: str = Field(..., min_length=1, max_length=200, description="Titre du livre")
    author: str = Field(..., min_length=1, description="Auteur principal")
    year: int = Field(..., ge=1450, le=2100, description="Annee de publication")
    isbn: Optional[str] = Field(None, pattern=r"^\d{10}(\d{3})?$", description="ISBN-10 ou ISBN-13")

    @field_validator("title", "author")
    @classmethod
    def not_just_whitespace(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("ne peut pas etre vide")
        return v

# Test
b = Book(title="  Le Petit Prince  ", author="St-Exupery", year=1943)
print(f"Title strippe : {b.title!r}")
print(f"JSON         : {b.model_dump_json(indent=2)}")

# Validation echoue
from pydantic import ValidationError
try:
    Book(title="", author="X", year=1900)
except ValidationError as e:
    print(f"\nErreur validation (attendue) : {e.errors()[0]['msg']}")

## 3. API CRUD complete — livres (testee in-process)


In [ ]:
from fastapi import HTTPException, status

app = FastAPI(title="Books API", version="1.0.0")

class BookIn(BaseModel):
    title: str = Field(..., min_length=1, max_length=200)
    author: str = Field(..., min_length=1)
    year: int = Field(..., ge=1450, le=2100)

class BookOut(BookIn):
    id: int

# Storage in-memory (en prod -> Postgres + SQLAlchemy / SQLModel)
db: dict[int, BookOut] = {}
_next_id = 0

@app.get("/books", response_model=list[BookOut], tags=["books"])
def list_books():
    """Liste tous les livres."""
    return list(db.values())

@app.get("/books/{book_id}", response_model=BookOut, tags=["books"])
def get_book(book_id: int):
    """Recupere un livre par son ID."""
    if book_id not in db:
        raise HTTPException(status.HTTP_404_NOT_FOUND, detail="Book not found")
    return db[book_id]

@app.post("/books", response_model=BookOut,
          status_code=status.HTTP_201_CREATED, tags=["books"])
def create_book(book: BookIn):
    """Cree un nouveau livre."""
    global _next_id
    _next_id += 1
    new = BookOut(id=_next_id, **book.model_dump())
    db[_next_id] = new
    return new

@app.put("/books/{book_id}", response_model=BookOut, tags=["books"])
def update_book(book_id: int, book: BookIn):
    """Met a jour completement un livre."""
    if book_id not in db:
        raise HTTPException(status.HTTP_404_NOT_FOUND, detail="Book not found")
    updated = BookOut(id=book_id, **book.model_dump())
    db[book_id] = updated
    return updated

@app.delete("/books/{book_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["books"])
def delete_book(book_id: int):
    """Supprime un livre."""
    if book_id not in db:
        raise HTTPException(status.HTTP_404_NOT_FOUND, detail="Book not found")
    del db[book_id]

client = TestClient(app)

# === Test du CRUD ===
print("=== CRUD test ===")

# CREATE
r = client.post("/books", json={"title": "1984", "author": "Orwell", "year": 1949})
print(f"POST /books         : {r.status_code}  body={r.json()}")
book_id = r.json()["id"]

# LIST
print(f"GET  /books         : {len(client.get('/books').json())} books")

# READ
print(f"GET  /books/{book_id}     : {client.get(f'/books/{book_id}').json()}")

# UPDATE
client.put(f"/books/{book_id}", json={"title": "1984 (rev)", "author": "Orwell", "year": 1949})
print(f"PUT  /books/{book_id}     : {client.get(f'/books/{book_id}').json()['title']}")

# DELETE
print(f"DELETE /books/{book_id}  : {client.delete(f'/books/{book_id}').status_code}")
print(f"GET  /books/{book_id}     : {client.get(f'/books/{book_id}').status_code} (404 attendu)")

## 4. Async vs sync — quand utiliser quoi ?

| Cas | Recommandation |
|---|---|
| Endpoint qui appelle `requests.get` / Postgres synchrone | `def` (sync) — FastAPI le lance dans threadpool |
| Endpoint qui appelle `httpx.AsyncClient` / `asyncpg` / `aiofiles` | `async def` |
| Endpoint pur CPU (calcul ML) | `def` (sync) — async ne sert a rien sur CPU bound |
| Endpoint qui attend du I/O (DB, HTTP outbound) | `async def` — gain enorme |

> ⚠️ **JAMAIS** de `time.sleep()` ou `requests.get()` dans un `async def` : bloque le event loop. Utiliser `asyncio.sleep`, `httpx.AsyncClient`.


In [ ]:
import asyncio

app2 = FastAPI()

@app2.get("/sync")
def sync_endpoint():
    """Sync : tourne dans un threadpool (FastAPI gere)."""
    return {"mode": "sync", "tid": "via threadpool"}

@app2.get("/async")
async def async_endpoint():
    """Async : tourne dans le event loop principal."""
    await asyncio.sleep(0.01)
    return {"mode": "async", "tid": "event loop"}

c2 = TestClient(app2)
print(f"GET /sync  : {c2.get('/sync').json()}")
print(f"GET /async : {c2.get('/async').json()}")

## 5. `Depends` — injection de dependances

Permet de **factoriser** : auth, DB session, parametres de pagination communs, etc.


In [ ]:
from fastapi import Depends, Header, Query

app3 = FastAPI()

# Pagination commune (reutilisable)
def common_pagination(
    skip: int = Query(0, ge=0, description="Items a skipper"),
    limit: int = Query(10, ge=1, le=100, description="Max items"),
) -> dict:
    return {"skip": skip, "limit": limit}

@app3.get("/items")
def list_items(pagination: dict = Depends(common_pagination)):
    return {"data": ["item_a", "item_b", "item_c"], **pagination}

# Auth simple via header
def get_user(x_token: str = Header(...)):
    if x_token != "secret-token":
        raise HTTPException(401, "Invalid token")
    return {"user_id": 42, "name": "alice"}

@app3.get("/me")
def me(user: dict = Depends(get_user)):
    return user

c3 = TestClient(app3)
print(f"GET /items?limit=3      : {c3.get('/items?limit=3').json()}")
print(f"GET /me (sans token)    : {c3.get('/me').status_code} (422)")
print(f"GET /me (bad token)     : {c3.get('/me', headers={'X-Token': 'wrong'}).status_code} (401)")
print(f"GET /me (good token)    : {c3.get('/me', headers={'X-Token': 'secret-token'}).json()}")

## 6. OAuth2 + JWT — pattern recommande

Pattern : `POST /token` avec username/password → renvoie JWT. Endpoint protege requiert `Authorization: Bearer <jwt>`.


In [ ]:
# pip install -q python-jose[cryptography] passlib[bcrypt] python-multipart
try:
    from datetime import datetime, timedelta, timezone
    from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
    from jose import jwt, JWTError
    from passlib.context import CryptContext

    SECRET_KEY = "demo-secret-changeme-in-prod"
    ALGO = "HS256"

    # On utilise pbkdf2_sha256 plutot que bcrypt (qui a une limite 72 bytes
    # et un bug connu avec passlib + bcrypt 4.0+).
    # En prod, prefere argon2 ou bcrypt si tu maitrises la version.
    pwd_ctx = CryptContext(schemes=["pbkdf2_sha256"])
    oauth2 = OAuth2PasswordBearer(tokenUrl="token")

    fake_users = {
        "alice": {"username": "alice", "hashed_password": pwd_ctx.hash("wonderland")},
    }

    def create_token(data: dict, expires: timedelta = timedelta(minutes=30)):
        to_encode = data.copy()
        to_encode["exp"] = datetime.now(timezone.utc) + expires
        return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGO)

    app4 = FastAPI()

    @app4.post("/token")
    def login(form: OAuth2PasswordRequestForm = Depends()):
        user = fake_users.get(form.username)
        if not user or not pwd_ctx.verify(form.password, user["hashed_password"]):
            raise HTTPException(401, "Bad credentials")
        return {"access_token": create_token({"sub": form.username}), "token_type": "bearer"}

    def current_user(token: str = Depends(oauth2)):
        try:
            payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGO])
            username = payload.get("sub")
            if username is None: raise JWTError()
        except JWTError:
            raise HTTPException(401, "Invalid token")
        return fake_users[username]

    @app4.get("/me")
    def me_jwt(user: dict = Depends(current_user)):
        return {"username": user["username"]}

    c4 = TestClient(app4)
    # Login
    r = c4.post("/token", data={"username": "alice", "password": "wonderland"})
    print(f"POST /token : {r.status_code}")
    token = r.json()["access_token"]
    print(f"Token (40 premiers chars) : {token[:40]}...")

    # Acces protege
    r = c4.get("/me", headers={"Authorization": f"Bearer {token}"})
    print(f"GET /me (good JWT) : {r.json()}")

    r = c4.get("/me", headers={"Authorization": "Bearer bad-token"})
    print(f"GET /me (bad JWT)  : {r.status_code}")
except ImportError as e:
    print(f"Optional deps manquantes : {e}")
    print("Installer : pip install -q python-jose[cryptography] passlib[bcrypt] python-multipart")

## 7. Tests automatises (pytest + TestClient)

Pattern recommande : ecrire les tests **comme client API** (HTTP), pas en testant les fonctions internes. Permet de detecter les regressions sur le contrat HTTP.


In [ ]:
# test_books.py (fichier separe en realite)

# === Tests inline (simule pytest) ===
def test_create_book_returns_201():
    r = client.post("/books", json={"title": "T", "author": "A", "year": 2020})
    assert r.status_code == 201
    assert "id" in r.json()

def test_get_404():
    assert client.get("/books/99999").status_code == 404

def test_invalid_payload_returns_422():
    r = client.post("/books", json={"title": "", "author": "A", "year": 2020})
    # FastAPI renvoie 422 automatiquement si validation Pydantic echoue
    assert r.status_code == 422

def test_year_out_of_range():
    r = client.post("/books", json={"title": "X", "author": "A", "year": 3000})
    assert r.status_code == 422
    errors = r.json()["detail"]
    assert any("year" in str(err.get("loc", [])) for err in errors)

# Reset db pour les tests
db.clear()

# Run inline
for name, fn in list(globals().items()):
    if name.startswith("test_") and callable(fn):
        try:
            fn()
            print(f"  PASS  {name}")
        except AssertionError as e:
            print(f"  FAIL  {name} : {e}")

## 8. Serving ML — pattern lifespan + state

Pattern : **charger le modele au demarrage** (pas a chaque requete), garder en memoire via `lifespan` + `state`.


In [ ]:
from contextlib import asynccontextmanager
from fastapi import FastAPI

ml_state: dict = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    """S'execute une fois au demarrage et une fois a l'arret."""
    # Startup : charger le modele
    print("Loading model...")
    ml_state["model"] = lambda x: sum(x)         # fake model = somme des features
    ml_state["version"] = "1.0.0"
    yield
    # Shutdown : cleanup
    ml_state.clear()
    print("Model unloaded.")

app_ml = FastAPI(lifespan=lifespan)

class Features(BaseModel):
    x: list[float] = Field(..., min_length=1, max_length=100)

class Prediction(BaseModel):
    score: float
    model_version: str

@app_ml.post("/predict", response_model=Prediction)
def predict(features: Features):
    model = ml_state["model"]
    return Prediction(score=float(model(features.x)),
                     model_version=ml_state["version"])

@app_ml.get("/health")
def health():
    """Readiness probe pour K8s."""
    return {"status": "ok" if "model" in ml_state else "loading"}

with TestClient(app_ml) as c_ml:
    print(f"GET /health  : {c_ml.get('/health').json()}")
    r = c_ml.post("/predict", json={"x": [1.5, 2.5, 3.0]})
    print(f"POST /predict (sum [1.5, 2.5, 3.0]) : {r.json()}")

## 9. Deploiement

### Dev local
```bash
uvicorn main:app --reload
```

### Production (recommande) : Gunicorn + Uvicorn workers
```bash
pip install gunicorn
gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000
```

### Dockerfile minimal
```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
```

### Reverse-proxy Nginx (TLS terminaison)
```nginx
server {
    listen 80;
    server_name api.example.com;
    location / {
        proxy_pass http://127.0.0.1:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    }
}
```

### Plateformes managees
- **Render**, **Fly.io**, **Railway** : git push → deploye
- **Hugging Face Spaces** : 16 GB RAM gratuit, ideal pour API + modele ML
- **AWS Lambda** : via `mangum` adapter


## 10. Comparatif Flask vs FastAPI

| Critere | Flask | FastAPI |
|---|---|---|
| Sync only | ✅ | Sync + async |
| Validation | Manuelle (marshmallow / pydantic externe) | Native via Pydantic |
| OpenAPI auto | Non (flasgger en plugin) | Oui (gratuit) |
| Docs interactives | Non par defaut | Swagger + ReDoc gratuit |
| Type-hint friendly | Limite | Source de verite |
| Performance | ~ 2-3k req/s | ~ 5-15k req/s |
| Maturite ecosysteme | Enorme (depuis 2010) | Solide (depuis 2018) |

> 🎯 **Recommandation 2025** : nouveau projet → **FastAPI**. Maintenance legacy → Flask OK.


## 11. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| `time.sleep()` ou `requests.get()` dans `async def` | `asyncio.sleep`, `httpx.AsyncClient` |
| Garder `debug=True` en prod | `debug=False`, gerer via env var |
| Pas de validation Pydantic (laisser dict) | Toujours typer le payload |
| Mot de passe en clair dans le code | `passlib.bcrypt`, secret en env var |
| `SECRET_KEY` hardcodee | Variable d'environnement |
| Charger le modele a chaque request | Lifespan + state |
| Pas de health check `/health` | Obligatoire pour K8s readiness |
| Logger uniquement avec `print` | `logging.getLogger(__name__)` structurè |
| Pas de rate limiting | `slowapi` (FastAPI) |
| OpenAPI public en prod | Restreindre `/docs` aux env de dev |
| Pas de CORS configure | `CORSMiddleware` avec allowlist |
| Pas de tests | `pytest` + `TestClient` |


## 12. References

### Documentation officielle
- **FastAPI** : https://fastapi.tiangolo.com/
- **Pydantic v2** : https://docs.pydantic.dev/latest/
- **Uvicorn** : https://www.uvicorn.org/
- **Starlette** : https://www.starlette.io/

### Patterns
- *Full Stack FastAPI Template* (officiel) : https://github.com/fastapi/full-stack-fastapi-template
- *SQLModel* (FastAPI + SQLAlchemy) : https://sqlmodel.tiangolo.com/

### Voir aussi (collection)
- [Flask_API.ipynb](./Flask_API.ipynb) — alternative classique
- [MLOps_Model_Serving.ipynb](./MLOps_Model_Serving.ipynb) — patterns prod (BentoML, Triton)
- [Streamlit_brique.ipynb](./Streamlit_brique.ipynb) — UI rapide pour appeler une API
